In [ ]:
from pyspark.sql.functions import lit, col, isnan, when, expr, year, count
from pyspark.sql import SparkSession

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Data Cleansing") \
    .getOrCreate()


In [ ]:
# Load the CSV file into a Spark DataFrame
file_path = "scopus_300_search.csv"
df = spark.read.csv(file_path, header=True, inferSchema=True)

# Show initial data
df.show()


In [ ]:
# Drop rows where any of the specified columns have missing values
columns_to_check = [
    "Subject", "URL", "Identifier", "EID", "Title", 
    "Creator", "Publication Name", "DOI", "Affiliation Name", 
    "Affiliation City", "Affiliation Country"
]

df = df.na.drop(subset=columns_to_check)

# Show the resulting DataFrame
df.show()

# Count the number of rows after dropping
print(f"Number of rows after dropping missing values: {df.count()}")


In [ ]:
# Add a new column 'year' with the value 2024
df = df.withColumn("Year", lit(2024))

# Show the updated DataFrame
df.show()

In [ ]:
# Count all unique values in the 'Affiliation Country' column
unique_countries_count = df.select("Affiliation Country").distinct().count()

print(f"Number of unique countries: {unique_countries_count}")


In [ ]:
# Drop the 'URL' column from the DataFrame
df = df.drop("URL")

# Show the updated DataFrame
df.show()


In [ ]:
# Ensure 'Publication Date' is in a proper date format
df = df.withColumn("Publication Date", col("Publication Date").cast("date"))
print(f"Number of rows before filtering: {df.count()}")

# Drop rows where 'Publication Date' is not in 2024
df = df.filter(year(col("Publication Date")) == 2024)

# Show the filtered DataFrame
df.show()

# Count the number of remaining rows
print(f"Number of rows remaining after filtering: {df.count()}")

In [ ]:
# Count NULLs in 'Publication Date'
null_pub_date = df.filter(col("Publication Date").isNull()).count()

# Count NULLs in 'Cited By Count'
null_cited_by_count = df.filter(col("Cited By Count").isNull()).count()

print(f"Number of NULLs in 'Publication Date': {null_pub_date}")
print(f"Number of NULLs in 'Cited By Count': {null_cited_by_count}")

In [ ]:
df.show()

In [ ]:
# Save DataFrame to CSV
import os

output_path = os.path.abspath("cleaned_308_researches.csv")
df.write.csv(output_path, header=True, mode="overwrite")

print(f"DataFrame saved to {output_path}")
